# Módulo 7 — Cointegração
## Seção 7.4 — Modelos de correção de erros (ECM): dedução, estimação e interpretação

**Estado:** material didático para revisão científica. **Pré-requisitos:** Seções 7.2–7.3; equações de diferenças, AR(1), MQO e interpretação de regressão. **Pergunta central:** como uma relação de longo prazo influencia as variações de curto prazo?

**Rastreabilidade:** Morettin e Toloi (2018), cap. 5, §5.3.2, e Apêndice B fornecem a base univariada sobre processos integrados e diferenças. ECM multivariado é **complementação externa** associada a Engle e Granger (1987), sem atribuição de páginas específicas à obra univariada.

### 1. Intuição: dois mecanismos operam simultaneamente

Suponha que $Y_t$ e $X_t$ sejam $I(1)$, mas $u_t=Y_t-\alpha-\beta X_t$ seja $I(0)$. O desvio $u_{t-1}$ mede o afastamento do equilíbrio no período anterior. Um ECM reúne (i) **mudanças imediatas**, por diferenças contemporâneas ou defasadas, e (ii) **correção do desvio anterior**.

Uma forma condicional é

$$\boxed{\Delta Y_t=c+\lambda_{EC}\,(Y_{t-1}-\alpha-\beta X_{t-1})+\theta\,\Delta X_t+\sum_{j=1}^{p}\varphi_j\Delta Y_{t-j}+\sum_{j=1}^{q}\psi_j\Delta X_{t-j}+\varepsilon_t.}$$

$\lambda_{EC}$ é o coeficiente de ajuste; $\beta$ é a inclinação de longo prazo; $\theta$ mede resposta contemporânea condicional de curto prazo; as demais inclinações captam dinâmica defasada. **O parâmetro $\lambda_{EC}$ não é o $\lambda$ de Box–Cox.**

### 2. Dedução exata a partir de um mecanismo estacionário

Escolha $X_t=X_{t-1}+\xi_t$, $Y_t=\alpha+\beta X_t+u_t$, $u_t=\phi u_{t-1}+\eta_t$, com $|\phi|<1$ e $u_t$ na solução estacionária. Diferenciando $Y_t$:

$$\Delta Y_t=\beta\Delta X_t+u_t-u_{t-1}.$$

Reorganizando a dinâmica de $u_t$:

$$u_t-u_{t-1}=(\phi-1)u_{t-1}+\eta_t.$$

Substituindo $u_{t-1}=Y_{t-1}-\alpha-\beta X_{t-1}$:

$$\boxed{\Delta Y_t=\beta\Delta X_t-(1-\phi)(Y_{t-1}-\alpha-\beta X_{t-1})+\eta_t.}$$

Esta identidade **não** resulta de aproximação. Implica $\lambda_{EC}=\phi-1$. Para $0<\phi<1$, temos $-1<\lambda_{EC}<0$; para $-1<\phi<0$, o coeficiente fica entre $-2$ e $-1$, correspondendo a correção oscilatória. No modelo escalar isolado, a estabilidade exata é $|1+\lambda_{EC}|<1$, ou $-2<\lambda_{EC}<0$; **não** é correto impor universalmente $-1<\lambda_{EC}<0$.

**Exemplo numérico.** Para $\alpha=3$, $\beta=1{,}2$, $\phi=0{,}6$, $u_{t-1}=2$, $\Delta X_t=0{,}5$ e $\eta_t=0$, resulta

$$\Delta Y_t=1{,}2(0{,}5)+(0{,}6-1)(2)=0{,}6-0{,}8=-0{,}2.$$

Embora $X$ aumente, a correção do afastamento positivo domina a resposta no exemplo. O desvio seguinte será $u_t=0{,}6(2)=1{,}2$.

### 3. ECM e regressão de duas etapas

Se $\widehat\alpha,\widehat\beta$ foram obtidos pela relação de longo prazo, construa

$$\widehat{ECT}_{t-1}=Y_{t-1}-\widehat\alpha-\widehat\beta X_{t-1}.$$

Na segunda etapa, estime uma regressão para $\Delta Y_t$ com $\widehat{ECT}_{t-1}$ e diferenças relevantes. Defasagens devem ter justificativa metodológica e validação residual. **A inferência convencional que ignora que $\widehat{ECT}$ foi estimado pode ser inadequada**, especialmente em amostras pequenas; para aplicações científicas devem ser avaliadas inferências apropriadas ao modelo e às condições assintóticas, diagnóstico e possíveis endogeneidades.

Se $\Delta X_t$ estiver correlacionada com o erro da equação, MQO simples não terá interpretação causal nem necessariamente consistência. O ECM condicional exige uma especificação adequada de exogeneidade; num sistema em que ambas as variáveis se ajustam, uma representação VECM é mais apropriada (Seção 7.6).

### 4. Simulação e estimação por MQO

O experimento usa o mesmo mecanismo matematicamente construído. Comparamos o estimador com os parâmetros geradores, mas **não** tratamos simulações como comprovação geral.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

rng = np.random.default_rng(742)
n = 750
alfa, beta, phi = 3.0, 1.2, 0.6
x = np.cumsum(rng.normal(size=n))
u = np.zeros(n)
eta = rng.normal(scale=0.55, size=n)
for t in range(1, n):
    u[t] = phi*u[t-1] + eta[t]
y = alfa + beta*x + u

long_run = sm.OLS(y, sm.add_constant(x)).fit()
a_hat, b_hat = long_run.params
ect = y - (a_hat + b_hat*x)
dy = np.diff(y)
dx = np.diff(x)
# ECT_{t-1} tem índices 0,...,n-2; diferenças têm índices 1,...,n-1
X_ecm = sm.add_constant(np.column_stack([ect[:-1], dx]))
fit = sm.OLS(dy, X_ecm).fit()
print('Longo prazo (alfa,beta):', np.round([a_hat,b_hat], 3))
print('ECM (constante, ajuste, resposta de curto prazo):', np.round(fit.params,3))
print('Parâmetros verdadeiros de ajuste e curto prazo:', [phi-1,beta])

fig, axes = plt.subplots(2,1,figsize=(9,5.5),sharex=True)
axes[0].plot(ect, label='Desvio de equilíbrio estimado')
axes[0].axhline(0, color='black', lw=0.8); axes[0].legend()
axes[1].scatter(ect[:-1],dy-beta*dx,s=10,alpha=0.4,label='Variação descontada do efeito de ΔX')
z=np.linspace(np.quantile(ect,0.01),np.quantile(ect,0.99),100)
axes[1].plot(z,(phi-1)*z,label='Inclinação teórica = phi−1')
for ax in axes: ax.grid(alpha=0.25);ax.legend()
axes[1].set_xlabel('Desvio defasado'); fig.tight_layout()
plt.savefig('figuras/07_04_ajuste_ecm.png', dpi=145)
plt.show()

**Figura 7.4.1 —** Série de desvios estimados e relação entre o desvio defasado e a variação líquida da influência contemporânea de $\Delta X_t$. A nuvem reflete o ruído $\eta_t$ e a estimação da relação de longo prazo.

### 5. Interpretação estatística cuidadosa

- O coeficiente de ajuste negativo representa retorno ao equilíbrio **somente conforme a definição e sinal adotados para o ECT**.
- O intercepto no ECM depende da parametrização e de onde os termos determinísticos são colocados; não deve ser somado indiscriminadamente em duas equações.
- O coeficiente $\theta$ contemporâneo não estabelece causalidade. Na prática, hipóteses de exogeneidade e simultaneidade importam.
- Os resíduos da segunda regressão precisam ser avaliados: autocorrelação, heterocedasticidade, mudanças de regime e estabilidade paramétrica.
- Um teste de cointegração significativo não garante previsões melhores que modelos alternativos; é necessária avaliação fora da amostra e respeito à ordem temporal.
- Com $\lambda_{EC}=0$ não há correção **pela equação considerada**. Em um sistema multivariado, outra variável pode ser a responsável pelo ajuste.

### 6. Exercícios de autonomia

**A.** Reconstrua a dedução do ECM para $u_t=0{,}8u_{t-1}+\eta_t$. Calcule o desvio após três períodos sem novos choques partindo de $u_0=5$.

**B.** Analise o caso $\phi=-0{,}4$. Qual é o coeficiente de ajuste? O que significa a alternância de sinais do desvio?

**C.** Construa o pseudocódigo para estimar a relação de longo prazo, calcular `ECT[:-1]` e formar diferenças. Explique os alinhamentos de índices antes de programar.

**D.** Por que a regressão de $\Delta Y_t$ sobre ECT estimado pode exigir cuidados inferenciais? Quais diagnósticos devem vir antes de usar o modelo para previsão?

### Referências

**Base univariada:** Morettin; Toloi (2018), cap. 5, §§5.3.1–5.3.2, pp. 142–145 e Apêndice B. **Fonte específica complementar:** Engle; Granger (1987), *Econometrica*, 55(2), 251–276; e a formulação geral de sistemas VAR/VECM em Johansen (1991). As simulações e a dedução do exemplo são complementações didáticas, não transcrições do livro.